# LV 几何 AHA 17 分区可视化（v3：稳定 shape=(1,2) + 底部 legend）

## 相比 v2 的关键修正

v2 给每个视图**独立**开一个 `pv.Plotter(off_screen=True)`，连续创建关闭的 off-screen plotter 在部分后端（EGL / OSMesa / 远程 xvfb）下会出现 **OpenGL context 复用不稳** 的问题，表现为 screenshot 返回空白图 —— 也就是你看到的"只有背景没有 3D 内容"。

v3 的修正思路：

1. **回到单个 `pv.Plotter(shape=(1,2))`**  
   v1 的 cutaway 能成功说明这个结构在你的环境下是稳定的。只开一次 context，彻底避开连续多 plotter 的问题。
2. **彻底关闭 PyVista 内嵌 scalar bar**  
   v1 的"外表面没出来"其实就是 `scalar_bar_args.position_x=0.88` 在双子图下错位遮挡导致的，v3 直接不在 PyVista 里画 scalar bar。
3. **加 `pl.reset_camera()` 修正相机 fit** ⭐ 关键  
   原代码在 `add_mesh` 后直接 `camera_position = [...]`，这只改方向不重算 `parallel_scale`。在平行投影下会导致 mesh 被挤出视野（观察到的症状：整张图是 mesh 的一个 cell 的颜色 / 或者看不到 mesh 只剩背景）。加 `reset_camera()` 让 VTK 按当前方向自动重新 fit。
4. **两步 off-screen 截图**  
   先 `pl.show(auto_close=False)` 强制触发完整渲染，再 `pl.screenshot(transparent_background=...)`，这是 off-screen 模式下最稳的 API 顺序。
5. **matplotlib 只负责加底部 legend**  
   不再用 matplotlib 做左右拼接（这步交给 PyVista 的 shape=(1,2)），只在 3D 视图下方画 17 段离散 legend。
6. **前置一个 PyVista 环境自检 cell**  
   渲染一个简单红球测试离屏截图是否工作。如果红球都看不到，问题就出在底层而不是本 notebook 的代码。

数据输入不变：Abaqus 六面体 `.inp` + MATLAB 分区 `.mat`。

## 运行前准备

```bash
pip install pyvista vtk scipy matplotlib h5py
```

Linux 服务器 / 容器若无显示服务器：

```bash
sudo apt-get install -y xvfb libgl1-mesa-glx
```

In [ ]:
import os
from pathlib import Path
import tempfile

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from scipy.io import loadmat

try:
    import h5py   # 仅用于 MATLAB v7.3 文件回退读取
except Exception:
    h5py = None

import pyvista as pv

# headless 环境尝试启动 xvfb（macOS/Windows 会静默失败）
try:
    pv.start_xvfb()
except Exception:
    pass

# 统一 PyVista 背景
pv.global_theme.background = "white"

print("PyVista version :", pv.__version__)
print("VTK version     :", pv.vtk_version_info)


## 0. PyVista 环境自检

**必须先运行这个 cell**。如果下面显示不出红色球，说明当前 Jupyter 内核的 PyVista 离屏渲染本身就有问题，**继续运行后面的 cell 也不会出图**。此时应检查：

- 是不是用 `xvfb-run jupyter notebook` 启动的
- Linux 服务器是否装了 `libgl1-mesa-glx`
- 远程容器的 OpenGL 后端是否能用（试 `MESA_GL_VERSION_OVERRIDE=3.3`）

In [ ]:
# PyVista 离屏渲染自检
pl_test = pv.Plotter(off_screen=True, window_size=(400, 400))
pl_test.set_background("lightgray")
pl_test.add_mesh(pv.Sphere(radius=1.0), color="red", show_edges=True)
pl_test.camera_position = "iso"

# 关键：show(auto_close=False) 先触发渲染，再 screenshot
pl_test.show(auto_close=False)
img_test = pl_test.screenshot(filename=None, return_img=True)
pl_test.close()

print(f"Screenshot shape: {img_test.shape}, dtype: {img_test.dtype}")
# lightgray ≈ (211, 211, 211)；若图里有明显红色像素，渲染就是 OK 的
r, g, b = img_test[..., 0], img_test[..., 1], img_test[..., 2]
red_like = (r > 200) & (g < 100) & (b < 100)
print(f"Red-ish pixels  : {red_like.sum()} (should be > 0)")

plt.figure(figsize=(3, 3))
plt.imshow(img_test)
plt.axis("off")
plt.title("PyVista offscreen self-test")
plt.show()


In [ ]:
# =========================
# 1) 用户配置：只需修改这里
# =========================
inp_path = "hexheartLVmeshF60S45.inp"
mat_path = "AHALVMeshDivisionPCAReconstructed.mat"

# 输出路径
output_dir = "output_dir"
figure_name = "lv_aha17_pyvista_academic.png"
vtu_name = "lv_aha17_mesh.vtu"
save_vtu = True

# 背景模式："white" 或 "transparent"
bg_mode = "white"

# 图像参数
window_size = (2400, 1100)   # 双视图总渲染窗口（shape=(1,2)：左 1200×1100，右 1200×1100）
show_edges_left = False
show_edges_right = True
parallel_projection = True
mpl_dpi = 200


In [ ]:
# =========================
# 2) AHA 17 分区名称与颜色
# =========================
REGION_NAMES = {
    1: "Basal inferoseptal",
    2: "Basal anteroseptal",
    3: "Basal anterior",
    4: "Basal anterolateral",
    5: "Basal inferolateral",
    6: "Basal inferior",
    7: "Mid inferoseptal",
    8: "Mid anteroseptal",
    9: "Mid anterior",
    10: "Mid anterolateral",
    11: "Mid inferolateral",
    12: "Mid inferior",
    13: "Apical septal",
    14: "Apical anterior",
    15: "Apical lateral",
    16: "Apical inferior",
    17: "Apical cap",
}

# 17 个高区分度颜色
REGION_COLORS = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b",
    "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#aec7e8", "#ffbb78",
    "#98df8a", "#ff9896", "#c5b0d5", "#c49c94", "#f7b6d2",
]

AHA_CMAP = ListedColormap(REGION_COLORS, name="aha17")
AHA_ANNOTATIONS = {float(i): str(i) for i in range(1, 18)}


In [ ]:
# =========================
# 3) 读取 .mat 和 .inp
# =========================
def _load_mat_fallback_v73(mat_file):
    """读取 MATLAB v7.3 (HDF5) 格式的 .mat 文件。"""
    if h5py is None:
        raise RuntimeError(
            "无法用 scipy.io.loadmat 读取该 .mat 文件，且当前环境没有 h5py。"
        )

    with h5py.File(mat_file, "r") as f:
        if "AHALVMeshDivision" not in f:
            raise KeyError("MAT 文件中没有找到 'AHALVMeshDivision' 结构体。")

        grp = f["AHALVMeshDivision"]

        def _read_array(name):
            arr = np.array(grp[name]).squeeze()
            return arr.reshape(-1)

        node_regions = _read_array("nodeRegions").astype(np.int32)
        el_regions = _read_array("elRegions").astype(np.int32)
        return node_regions, el_regions


def load_regions_from_mat(mat_file):
    """读取 nodeRegions 和 elRegions。"""
    try:
        mat = loadmat(mat_file, squeeze_me=True, struct_as_record=False)
        if "AHALVMeshDivision" not in mat:
            raise KeyError("MAT 文件中没有找到 'AHALVMeshDivision'。")

        s = mat["AHALVMeshDivision"]

        if hasattr(s, "nodeRegions"):
            node_regions = np.asarray(s.nodeRegions).reshape(-1).astype(np.int32)
            el_regions = np.asarray(s.elRegions).reshape(-1).astype(np.int32)
            return node_regions, el_regions

        node_regions = np.asarray(s["nodeRegions"]).reshape(-1).astype(np.int32)
        el_regions = np.asarray(s["elRegions"]).reshape(-1).astype(np.int32)
        return node_regions, el_regions

    except NotImplementedError:
        return _load_mat_fallback_v73(mat_file)


def parse_abaqus_inp(inp_file):
    """
    解析 Abaqus .inp 中的：
    - *Node
    - *Element, type=C3D8 / C3D8H
    """
    node_ids = []
    points = []
    elem_ids = []
    elem_conn_node_ids = []

    in_nodes = False
    in_elements = False
    element_is_hex8 = False

    with open(inp_file, "r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.strip()

            if not line or line.startswith("**"):
                continue

            if line.startswith("*"):
                upper = line.upper()
                in_nodes = upper.startswith("*NODE")
                in_elements = upper.startswith("*ELEMENT")
                element_is_hex8 = False
                if in_elements and ("TYPE=C3D8" in upper or "TYPE=C3D8H" in upper):
                    element_is_hex8 = True
                continue

            if in_nodes:
                parts = [p.strip() for p in line.split(",") if p.strip()]
                if len(parts) >= 4:
                    node_ids.append(int(parts[0]))
                    points.append([float(parts[1]), float(parts[2]), float(parts[3])])
            elif in_elements and element_is_hex8:
                parts = [p.strip() for p in line.split(",") if p.strip()]
                if len(parts) < 9:
                    raise ValueError(f"六面体单元连通性读取失败，当前行：{line}")
                elem_ids.append(int(parts[0]))
                elem_conn_node_ids.append([int(x) for x in parts[1:9]])

    node_ids = np.asarray(node_ids, dtype=np.int64)
    points = np.asarray(points, dtype=np.float64)
    elem_ids = np.asarray(elem_ids, dtype=np.int64)
    elem_conn_node_ids = np.asarray(elem_conn_node_ids, dtype=np.int64)

    if node_ids.size == 0:
        raise RuntimeError("没有从 .inp 中解析到节点。")
    if elem_ids.size == 0:
        raise RuntimeError("没有从 .inp 中解析到 C3D8/C3D8H 单元。")

    # 向量化：查找表把 Abaqus 节点 ID 映射成 0-based 索引
    max_id = int(node_ids.max())
    lookup = np.full(max_id + 2, -1, dtype=np.int64)
    lookup[node_ids] = np.arange(node_ids.size, dtype=np.int64)

    conn = lookup[elem_conn_node_ids]
    if (conn < 0).any():
        missing = elem_conn_node_ids[conn < 0][:5]
        raise RuntimeError(f"单元引用了未定义的节点 ID，例如：{missing.tolist()}")

    return node_ids, points, elem_ids, conn


In [ ]:
# =========================
# 4) 构造 PyVista 网格
# =========================
def build_unstructured_grid(points, conn, node_regions, el_regions):
    if len(points) != len(node_regions):
        raise ValueError(
            f"节点数不一致：mesh 有 {len(points)} 个节点，"
            f"但 nodeRegions 长度为 {len(node_regions)}。"
        )
    if len(conn) != len(el_regions):
        raise ValueError(
            f"单元数不一致：mesh 有 {len(conn)} 个单元，"
            f"但 elRegions 长度为 {len(el_regions)}。"
        )

    n_cells = conn.shape[0]
    cells = np.hstack(
        [np.full((n_cells, 1), 8, dtype=np.int64), conn.astype(np.int64)]
    ).ravel()
    celltypes = np.full(n_cells, pv.CellType.HEXAHEDRON, dtype=np.uint8)

    grid = pv.UnstructuredGrid(cells, celltypes, points)
    grid.cell_data["AHA17"] = el_regions.astype(np.int32)
    grid.point_data["AHA17_nodes"] = node_regions.astype(np.int32)
    return grid


def maybe_save_vtu(grid, out_path):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    grid.save(out_path)
    return out_path


In [ ]:
# =========================
# 5) 渲染：shape=(1,2) + 底部 matplotlib legend
# =========================
def compute_camera(bounds):
    xmin, xmax, ymin, ymax, zmin, zmax = bounds
    center = np.array(
        [(xmin + xmax) / 2, (ymin + ymax) / 2, (zmin + zmax) / 2], dtype=float
    )
    extents = np.array([xmax - xmin, ymax - ymin, zmax - zmin], dtype=float)
    diag = np.linalg.norm(extents)
    position = center + np.array([1.35, -1.85, 1.05]) * diag
    viewup = (0.0, 0.0, 1.0)
    return [tuple(position), tuple(center), viewup]


def _draw_discrete_legend(ax, ncol=9, fontsize=9):
    handles = [
        mpatches.Patch(
            facecolor=REGION_COLORS[i],
            edgecolor="#333333",
            linewidth=0.6,
            label=f"{i + 1:>2d}  {REGION_NAMES[i + 1]}",
        )
        for i in range(17)
    ]
    ax.axis("off")
    ax.legend(
        handles=handles,
        loc="center",
        ncol=ncol,
        fontsize=fontsize,
        frameon=False,
        handlelength=1.2,
        handleheight=1.1,
        columnspacing=1.2,
        borderaxespad=0,
    )


def render_pyvista_dual(
    grid,
    window_size=(2400, 1100),
    transparent=False,
    zoom=1.1,
):
    """
    在一个 PyVista Plotter 里渲染 shape=(1,2) 双视图，返回像素数组。

    关键：
    - 只开一次 Plotter，避免连续创建 off-screen plotter 在部分后端下 OpenGL
      context 复用不稳的问题。
    - 关闭内嵌 scalar bar（v1 的外表面被遮挡就是 scalar_bar_args.position_x
      在双子图下错位导致的）。
    - 先 show(auto_close=False) 强制渲染，再 screenshot 取像素。
    """
    surface = grid.extract_surface()
    center = np.array(grid.center)
    clipped = grid.clip(normal=(1.0, 0.0, 0.0), origin=center, invert=False)
    clipped_surface = clipped.extract_surface()

    pl = pv.Plotter(
        shape=(1, 2),
        off_screen=True,
        window_size=window_size,
        border=False,
    )
    pl.set_background("white")

    camera = compute_camera(grid.bounds)

    panels = [
        (surface, "Exterior surface", show_edges_left),
        (clipped_surface, "Cutaway view", show_edges_right),
    ]
    for col, (mesh, title, show_edges) in enumerate(panels):
        pl.subplot(0, col)
        if parallel_projection:
            pl.enable_parallel_projection()
        pl.add_mesh(
            mesh,
            scalars="AHA17",
            preference="cell",
            cmap=AHA_CMAP,
            clim=(0.5, 17.5),
            categories=True,
            annotations=AHA_ANNOTATIONS,
            show_scalar_bar=False,     # 关键：关闭 PyVista 内嵌 scalar bar
            interpolate_before_map=False,
            show_edges=show_edges,
            edge_color="#333333",
            line_width=0.7,
            smooth_shading=False,
            ambient=0.18,
            diffuse=0.82,
            specular=0.05,
            lighting=True,
        )
        pl.add_text(title, position="upper_left", font_size=14, color="black")
        pl.camera_position = camera
        pl.reset_camera()          # 关键：按当前方向自动 fit parallel_scale / clip range
        pl.camera.zoom(zoom)

    # off-screen 最稳的两步截图：先 show 触发渲染，再 screenshot 拿像素
    pl.show(auto_close=False)
    img = pl.screenshot(
        filename=None,
        transparent_background=transparent,
        return_img=True,
    )
    pl.close()
    return img


def compose_with_legend(
    img_raw,
    screenshot_path,
    transparent=False,
    dpi=200,
    legend_ncol=9,
):
    """用 matplotlib 给 PyVista screenshot 在下方加 17 段离散 legend。"""
    fig_facecolor = "none" if transparent else "white"

    # 保持 PyVista 图的宽高比
    h, w = img_raw.shape[:2]
    aspect = w / h
    fig_h = 8.0
    fig_w = fig_h * aspect

    fig = plt.figure(figsize=(fig_w, fig_h + 1.2), facecolor=fig_facecolor)
    gs = fig.add_gridspec(
        2, 1,
        height_ratios=[fig_h, 1.2],
        hspace=0.04,
        left=0.01, right=0.99, top=0.99, bottom=0.01,
    )

    ax_img = fig.add_subplot(gs[0])
    ax_img.imshow(img_raw)
    ax_img.axis("off")

    ax_legend = fig.add_subplot(gs[1])
    _draw_discrete_legend(ax_legend, ncol=legend_ncol, fontsize=9)

    screenshot_path = Path(screenshot_path)
    screenshot_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        screenshot_path,
        dpi=dpi,
        bbox_inches="tight",
        facecolor=fig_facecolor,
        transparent=transparent,
    )
    plt.show()
    return screenshot_path


In [ ]:
# =========================
# 6) 读取数据并构造网格
# =========================
inp_path = Path(inp_path)
mat_path = Path(mat_path)
output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

figure_path = output_dir / figure_name
vtu_path = output_dir / vtu_name

node_regions, el_regions = load_regions_from_mat(mat_path)
node_ids, points, elem_ids, conn = parse_abaqus_inp(inp_path)

grid = build_unstructured_grid(points, conn, node_regions, el_regions)

print(f"Mesh points : {grid.n_points}")
print(f"Mesh cells  : {grid.n_cells}")
print(f"AHA labels  : {np.unique(grid.cell_data['AHA17'])}")

if save_vtu:
    maybe_save_vtu(grid, vtu_path)
    print(f"Saved VTU   : {vtu_path}")


In [ ]:
# =========================
# 7) AHA 编号 <-> 名称对照
# =========================
for k in range(1, 18):
    print(f"{k:>2d}  ->  {REGION_NAMES[k]}")


In [ ]:
# =========================
# 8) 生成双视图 + 底部 legend
# =========================
use_transparent = (bg_mode == "transparent")

# 先用 PyVista 渲出双视图
img_raw = render_pyvista_dual(
    grid,
    window_size=window_size,
    transparent=use_transparent,
    zoom=1.1,
)

print(f"Raw screenshot shape : {img_raw.shape}, dtype: {img_raw.dtype}")

# 快速健康检查：如果整张图都是背景色，给用户一个明确的提示
if img_raw.shape[-1] >= 3:
    # 背景是白色 (255,255,255)，要求至少有一定比例非白像素
    rgb = img_raw[..., :3]
    nonwhite_ratio = float((rgb.min(axis=-1) < 250).mean())
    print(f"Non-background pixel ratio: {nonwhite_ratio:.3f}")
    if nonwhite_ratio < 0.01:
        print("[WARN] 整张图几乎全是背景色 —— 请回到 cell 0 的 PyVista 自检确认渲染是否正常。")

# 再用 matplotlib 拼上底部 legend 并保存
compose_with_legend(
    img_raw,
    figure_path,
    transparent=use_transparent,
    dpi=mpl_dpi,
    legend_ncol=9,
)
print(f"Saved figure: {figure_path}")


## 可选：单独查看裁切后的几何

In [ ]:
# 单独渲染大尺寸 cutaway + 底部 legend
cutaway_path = output_dir / "lv_aha17_cutaway_only.png"

center = np.array(grid.center)
clipped = grid.clip(normal=(1.0, 0.0, 0.0), origin=center, invert=False)
clipped_surface = clipped.extract_surface()

pl = pv.Plotter(off_screen=True, window_size=(1500, 1400), border=False)
pl.set_background("white")
if parallel_projection:
    pl.enable_parallel_projection()

pl.add_mesh(
    clipped_surface,
    scalars="AHA17",
    preference="cell",
    cmap=AHA_CMAP,
    clim=(0.5, 17.5),
    categories=True,
    annotations=AHA_ANNOTATIONS,
    show_scalar_bar=False,
    show_edges=True,
    edge_color="#333333",
    line_width=0.7,
    ambient=0.18,
    diffuse=0.82,
    specular=0.05,
)
pl.add_text("Cutaway only", position="upper_left", font_size=14, color="black")
pl.camera_position = compute_camera(grid.bounds)
pl.reset_camera()          # 按当前方向自动 fit
pl.camera.zoom(1.15)

pl.show(auto_close=False)
img_cut = pl.screenshot(
    filename=None,
    transparent_background=use_transparent,
    return_img=True,
)
pl.close()

compose_with_legend(
    img_cut,
    cutaway_path,
    transparent=use_transparent,
    dpi=mpl_dpi,
    legend_ncol=5,
)
print(f"Saved figure: {cutaway_path}")


## 使用说明

运行顺序：

1. Cell 0（自检）—— **必须通过**。看到红色球就说明 PyVista 离屏渲染 OK
2. Cell 1（用户配置）—— 改 `inp_path` / `mat_path` / `output_dir` / `bg_mode`
3. 依次运行 Cell 2 ~ Cell 8
4. 如需单独看剖切视图，再运行最后一个可选单元

## 故障排查

**症状：Cell 0 的红色球看不到（整张图灰色）**

这说明当前 Jupyter 内核的 PyVista 离屏渲染本身就无法工作，和本 notebook 的代码无关。按顺序尝试：

1. 关掉 notebook，用 `xvfb-run -a jupyter notebook` 重启内核
2. `pip install --upgrade pyvista vtk`
3. Linux 容器确认 `libgl1-mesa-glx` 已安装
4. 设置环境变量：`export PYVISTA_OFF_SCREEN=true`

**症状：Cell 0 OK，但 Cell 8 图里外表面或剖切视图仍缺失**

1. 把 `window_size` 改小试试，比如 `(1600, 800)`，有些驱动在大 framebuffer 下会黑屏
2. 把 `show_edges_right = False`（大 mesh 画边线很吃显存）
3. 把 `parallel_projection = False` 改用透视投影

## 备注

- 按 `elRegions` 做 **cell-wise** 着色。
- 做 35 分区版本时把 `elRegions` 换成 `elRegionsFull`，并把 `REGION_NAMES` / `REGION_COLORS` 扩到 35 项，`clim=(0.5, 35.5)` 即可。